# HFCS data checks

Check the selected Parquet exports before constructing the estimation dataset. Household, person and derived files are assessed separately across all five imputations. Full diagnostic counts remain on the local data drive; only reduced summaries are written to the repository.


## Settings


In [1]:
from __future__ import annotations

import os
import re
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

WAVES = (2010, 2014, 2017, 2021, 2023)
IMPUTATIONS = (1, 2, 3, 4, 5)
TABLES = ("h", "d", "p")
MAX_CATEGORY_CODES = 12
MIN_FREQUENCY_TO_PUBLISH = 5
CATEGORICAL_PREFIXES = ("hh", "dh")

REPO_ROOT_OVERRIDE = None
WRITE_REPO_SUMMARIES = True


def discover_repo():
    explicit = REPO_ROOT_OVERRIDE or os.environ.get("HFCS_REPO_ROOT")
    if explicit:
        path = Path(explicit).expanduser().resolve()
        if not path.is_dir():
            raise FileNotFoundError(path)
        return path
    cwd = Path.cwd().resolve()
    for path in (cwd, *cwd.parents):
        if (path / "hfcs_file_inventory.txt").exists() or (path / ".git").exists():
            return path
    raise FileNotFoundError("Run from Research-Assistant-Git or set REPO_ROOT_OVERRIDE")


REPO = discover_repo()
INPUT = REPO.parent / "HFCS Data Download" / "HFCS_selected_raw"
PRIVATE_OUTPUT = INPUT / "diagnostics_private"
PUBLIC_OUTPUT = REPO / "HFCS" / "diagnostics"
if not INPUT.is_dir():
    raise FileNotFoundError(f"Missing Parquet folder: {INPUT}")
PRIVATE_OUTPUT.mkdir(parents=True, exist_ok=True)
if WRITE_REPO_SUMMARIES:
    PUBLIC_OUTPUT.mkdir(parents=True, exist_ok=True)
print("Parquet:", INPUT)
print("Detailed reports (local only):", PRIVATE_OUTPUT)
print("Repo summaries:", PUBLIC_OUTPUT if WRITE_REPO_SUMMARIES else "disabled")


Parquet: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester II\RA\HFCS Data Download\HFCS_selected_raw
Detailed reports (local only): C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester II\RA\HFCS Data Download\HFCS_selected_raw\diagnostics_private
Repo summaries: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester II\RA\Research-Assistant-Git\HFCS\diagnostics


## Input inventory

Expected input is 75 Parquet files: five waves, five imputations, and three tables (`h`, `d`, `p`). Run the export notebook first.


In [2]:
def parquet_path(wave, table, imp):
    return INPUT / f"hfcs_{wave}_{table}_imp{imp}.parquet"


def clean_name_map(names):
    result = {}
    for name in names:
        key = name.lower()
        if key in result and result[key] != name:
            raise ValueError(f"Duplicate column name ignoring case: {name}")
        result[key] = name
    return result


file_records, schema_records, issues = [], [], []
for wave in WAVES:
    for imp in IMPUTATIONS:
        for table in TABLES:
            path = parquet_path(wave, table, imp)
            entry = {
                "wave": wave, "table": table, "imputation": imp,
                "filename": path.name, "exists": path.is_file(),
                "rows": pd.NA, "raw_variables": pd.NA,
                "missing_code_companion_columns": pd.NA,
                "size_mb": round(path.stat().st_size / 1024**2, 2) if path.exists() else pd.NA,
            }
            if path.exists():
                parquet = pq.ParquetFile(path)
                names = parquet.schema_arrow.names
                real_names = [n for n in names if not n.lower().endswith("__stata_missing")]
                entry.update(rows=parquet.metadata.num_rows, raw_variables=len(real_names),
                             missing_code_companion_columns=len(names) - len(real_names))
                for field in parquet.schema_arrow:
                    schema_records.append({
                        "wave": wave, "table": table, "imputation": imp,
                        "variable": field.name, "parquet_type": str(field.type),
                        "stata_missing_companion": field.name.lower().endswith("__stata_missing"),
                    })
            else:
                issues.append({"wave": wave, "country": "ALL", "table": table,
                               "imputation": imp, "variable": "", "issue": "missing_parquet_file",
                               "detail": path.name})
            file_records.append(entry)

files_df = pd.DataFrame(file_records)
schema_df = pd.DataFrame(schema_records)
print(f"Files: {int(files_df['exists'].sum())} / {len(files_df)}")
if not files_df['exists'].all():
    raise FileNotFoundError("Missing exports. Run load_hfcs_updated.ipynb first; P files are now required.")
display(files_df.groupby(["wave", "table"], as_index=False)["exists"].agg(["sum", "count"]))


Files: 75 / 75


,wave,table,sum,count
0,2010,d,5,5
1,2010,h,5,5
2,2010,p,5,5
3,2014,d,5,5
4,2014,h,5,5
5,2014,p,5,5
6,2017,d,5,5
7,2017,h,5,5
8,2017,p,5,5
9,2021,d,5,5


## Variable checks

Check identifiers, missing values, Stata missing codes, and selected flag variables. Household IDs are allowed to repeat in the person file, where the person ID forms part of the key.


In [3]:
sample_rows, missing_rows, code_rows, extended_rows, text_rows, flag_rows = [], [], [], [], [], []

# Codes come from the HFCS User Guide, June 2026, boxes 3 and 4.
def flag_meaning(wave, raw_code):
    try:
        value = int(float(raw_code))
    except (TypeError, ValueError):
        return "unclassified"
    common = {0: "not_applicable", 1: "collected", 71: "register",
              1050: "not_imputed", 1053: "collected_in_brackets",
              3050: "edited", 4050: "imputed", 4053: "imputed_from_brackets",
              5050: "estimated_not_collected"}
    if value in common:
        return common[value]
    if wave <= 2017:
        if value == 2050:
            return "suppressed_for_anonymisation"
        if value == 2051:
            return "not_collected"
    else:
        if value == 2050:
            return "set_missing_reason_unspecified"
    return "other_flag"


def country_groups(df, country_col):
    return df.groupby(df[country_col].astype("string").fillna("<MISSING>"), sort=False, dropna=False)


def is_category(name, s):
    if not name.lower().startswith(CATEGORICAL_PREFIXES):
        return False
    if name.lower().startswith(("dhage", "dhbirth")):
        return False
    if not pd.api.types.is_numeric_dtype(s):
        return False
    values = s.dropna().unique()
    if not (0 < len(values) <= MAX_CATEGORY_CODES):
        return False
    try:
        arr = np.asarray(values, dtype=float)
        return bool(np.isfinite(arr).all() and np.isclose(arr, np.rint(arr)).all())
    except (ValueError, TypeError):
        return False


def summarize_group(wave, table, imp, country, sub, raw_columns, names):
    total = len(sub)
    household_id = names.get("sa0010")
    person_id = names.get("ra0010") if table == "p" else None
    key_cols = [c for c in (household_id, person_id) if c is not None]
    missing_keys = int(sub[key_cols].isna().any(axis=1).sum()) if key_cols else pd.NA
    duplicate_keys = int(sub.duplicated(key_cols, keep=False).sum()) if key_cols else pd.NA
    sample_rows.append({
        "wave": wave, "country": country, "table": table, "imputation": imp,
        "rows": total, "missing_household_id_rows": int(sub[household_id].isna().sum()) if household_id else pd.NA,
        "missing_key_rows": missing_keys, "duplicate_key_rows": duplicate_keys,
        "distinct_household_ids": int(sub[household_id].nunique()) if household_id else pd.NA,
        "raw_variables": len(raw_columns),
    })
    if not key_cols or (table == "p" and not person_id):
        issues.append({"wave": wave, "country": country, "table": table,
                       "imputation": imp, "variable": "keys",
                       "issue": "missing_identifier", "detail": "Person or household key is unavailable"})
    elif missing_keys or duplicate_keys:
        issues.append({"wave": wave, "country": country, "table": table,
                       "imputation": imp, "variable": "keys",
                       "issue": "duplicate_or_missing_identifier",
                       "detail": f"missing={missing_keys}; duplicate={duplicate_keys}"})

    for col in raw_columns:
        s = sub[col]
        missing_companion = names.get((col + "__stata_missing").lower())
        nulls = s.isna()
        extended = sub[missing_companion].notna() if missing_companion else pd.Series(False, index=sub.index)
        extension_n = int(extended.sum())
        missing_n = int((nulls | extended).sum())
        observed_n = total - missing_n
        overlap_n = int((~nulls & extended).sum())
        blank_n = 0
        replacement_n = 0
        control_n = 0
        if pd.api.types.is_string_dtype(s) or s.dtype == "object":
            present = s.dropna()
            if not present.empty and (pd.api.types.is_string_dtype(s) or isinstance(present.iloc[0], str)):
                text = present.astype("string")
                blank_n = int(text.str.fullmatch(r"\s*", na=False).sum())
                replacement_n = int(text.str.contains("\ufffd", regex=False, na=False).sum())
                control_n = int(text.str.contains(r"[\u0080-\u009f]", regex=True, na=False).sum())
        missing_rows.append({
            "wave": wave, "country": country, "table": table, "imputation": imp,
            "variable": col.lower(), "dtype": str(s.dtype), "rows": total,
            "observed_n": observed_n, "missing_total_n": missing_n,
            "missing_pct": round(100 * missing_n / total, 4) if total else pd.NA,
            "ordinary_null_n": int((nulls & ~extended).sum()),
            "stata_extended_missing_n": extension_n, "empty_string_n": blank_n,
            "unique_observed_n": int(s.nunique(dropna=True)),
        })
        if overlap_n:
            issues.append({"wave": wave, "country": country, "table": table,
                           "imputation": imp, "variable": col.lower(),
                           "issue": "stata_missing_companion_mismatch", "detail": f"{overlap_n} rows"})
        if replacement_n or control_n:
            text_rows.append({"wave": wave, "country": country, "table": table,
                              "imputation": imp, "variable": col.lower(),
                              "unicode_replacement_character_rows": replacement_n,
                              "unicode_control_character_rows": control_n})
        if missing_companion and extension_n:
            for value, count in sub.loc[extended, missing_companion].astype("string").value_counts().items():
                extended_rows.append({"wave": wave, "country": country, "table": table,
                                      "imputation": imp, "variable": col.lower(),
                                      "stata_missing_code": str(value), "count": int(count)})
        if col.lower() in ("fhh0100", "fra0300", "fra0300_b"):
            for flag, count in s.value_counts(dropna=False).items():
                flag_rows.append({"wave": wave, "country": country, "table": table,
                                  "imputation": imp, "variable": col.lower(),
                                  "flag": str(flag), "meaning": flag_meaning(wave, flag),
                                  "count": int(count)})
        if imp == 1 and total >= 50 and is_category(col, s):
            counts = s.value_counts(dropna=True)
            if counts.empty or (counts < MIN_FREQUENCY_TO_PUBLISH).any():
                continue
            for value, count in counts.items():
                code_rows.append({
                    "wave": wave, "country": country, "table": table,
                    "variable": col.lower(), "raw_numeric_code": str(value),
                    "count": int(count),
                    "pct_of_nonmissing": round(100 * count / observed_n, 3) if observed_n else pd.NA,
                })


for record in file_records:
    wave, table, imp = record["wave"], record["table"], record["imputation"]
    path = parquet_path(wave, table, imp)
    print("Checking", path.name, flush=True)
    frame = pd.read_parquet(path)
    names = clean_name_map(frame.columns)
    raw_columns = [x for x in frame.columns if not x.lower().endswith("__stata_missing")]
    country_col = names.get("sa0100")
    if country_col is None:
        issues.append({"wave": wave, "country": "ALL", "table": table,
                       "imputation": imp, "variable": "sa0100",
                       "issue": "missing_country_code", "detail": "Country column not exported"})
        continue
    if "im0100" in names:
        values = frame[names["im0100"]].dropna()
        if not values.empty and not values.astype("string").str.replace(r"\.0$", "", regex=True).eq(str(imp)).all():
            issues.append({"wave": wave, "country": "ALL", "table": table,
                           "imputation": imp, "variable": "im0100",
                           "issue": "incorrect_imputation", "detail": path.name})
    for country, sub in country_groups(frame, country_col):
        summarize_group(wave, table, imp, str(country), sub, raw_columns, names)
    del frame
    gc.collect()

samples_df = pd.DataFrame(sample_rows)
missing_df = pd.DataFrame(missing_rows)
flag_df = pd.DataFrame(flag_rows)
print("Country-file rows:", len(samples_df), "| Variable checks:", len(missing_df))


Checking hfcs_2010_h_imp1.parquet
Checking hfcs_2010_d_imp1.parquet
Checking hfcs_2010_p_imp1.parquet
Checking hfcs_2010_h_imp2.parquet
Checking hfcs_2010_d_imp2.parquet
Checking hfcs_2010_p_imp2.parquet
Checking hfcs_2010_h_imp3.parquet
Checking hfcs_2010_d_imp3.parquet
Checking hfcs_2010_p_imp3.parquet
Checking hfcs_2010_h_imp4.parquet
Checking hfcs_2010_d_imp4.parquet
Checking hfcs_2010_p_imp4.parquet
Checking hfcs_2010_h_imp5.parquet
Checking hfcs_2010_d_imp5.parquet
Checking hfcs_2010_p_imp5.parquet
Checking hfcs_2014_h_imp1.parquet
Checking hfcs_2014_d_imp1.parquet
Checking hfcs_2014_p_imp1.parquet
Checking hfcs_2014_h_imp2.parquet
Checking hfcs_2014_d_imp2.parquet
Checking hfcs_2014_p_imp2.parquet
Checking hfcs_2014_h_imp3.parquet
Checking hfcs_2014_d_imp3.parquet
Checking hfcs_2014_p_imp3.parquet
Checking hfcs_2014_h_imp4.parquet
Checking hfcs_2014_d_imp4.parquet
Checking hfcs_2014_p_imp4.parquet
Checking hfcs_2014_h_imp5.parquet
Checking hfcs_2014_d_imp5.parquet
Checking hfcs_

## Household identifiers

Check the household keys in the `h` and `d` files. A mismatch here would affect any subsequent join.


In [4]:
join_rows = []
KEYS = ("sa0100", "sa0010", "im0100")

for wave in WAVES:
    for imp in IMPUTATIONS:
        hp, dp = parquet_path(wave, "h", imp), parquet_path(wave, "d", imp)
        if not hp.exists() or not dp.exists():
            continue
        h_schema = clean_name_map(pq.ParquetFile(hp).schema_arrow.names)
        d_schema = clean_name_map(pq.ParquetFile(dp).schema_arrow.names)
        keys = [k for k in KEYS if k in h_schema and k in d_schema]
        if not {"sa0100", "sa0010"}.issubset(keys):
            issues.append({"wave": wave, "country": "ALL", "table": "h+d",
                           "imputation": imp, "variable": "keys",
                           "issue": "cannot_match_h_d", "detail": f"shared keys: {keys}"})
            continue
        h = pd.read_parquet(hp, columns=[h_schema[k] for k in keys])
        d = pd.read_parquet(dp, columns=[d_schema[k] for k in keys])
        h.columns = keys
        d.columns = keys
        h_dupes, d_dupes = int(h.duplicated(keys).sum()), int(d.duplicated(keys).sum())
        if h_dupes or d_dupes:
            issues.append({"wave": wave, "country": "ALL", "table": "h+d",
                           "imputation": imp, "variable": "keys",
                           "issue": "duplicate_h_d_merge_keys",
                           "detail": f"h={h_dupes}, d={d_dupes}"})
        # Distinct-key comparison, without multiplying matches from duplicates.
        merged = h.drop_duplicates().merge(d.drop_duplicates(), how="outer", on=keys,
                                            indicator=True, validate="one_to_one")
        for country, sub in country_groups(merged, "sa0100"):
            status = sub["_merge"].value_counts()
            only_h = int(status.get("left_only", 0))
            only_d = int(status.get("right_only", 0))
            both = int(status.get("both", 0))
            join_rows.append({"wave": wave, "country": str(country), "imputation": imp,
                              "matched_keys": both, "h_only_keys": only_h,
                              "d_only_keys": only_d})
            if only_h or only_d:
                issues.append({"wave": wave, "country": str(country), "table": "h+d",
                               "imputation": imp, "variable": "keys",
                               "issue": "h_d_identifier_mismatch",
                               "detail": f"h_only={only_h}, d_only={only_d}"})
        del h, d, merged
        gc.collect()

joins_df = pd.DataFrame(join_rows)
print("Household/derived join checks:", len(joins_df))
if not joins_df.empty:
    display(joins_df.head(12))


Household/derived join checks: 505


,wave,country,imputation,matched_keys,h_only_keys,d_only_keys
0,2010,AT,1,2380,0,0
1,2010,BE,1,2327,0,0
2,2010,CY,1,1237,0,0
3,2010,DE,1,3565,0,0
4,2010,E1,1,6197,0,0
5,2010,ES,1,6106,0,0
6,2010,FI,1,10989,0,0
7,2010,FR,1,15006,0,0
8,2010,GR,1,2971,0,0
9,2010,IT,1,7951,0,0


## Age of the reference person

The reference-person ID (`dhidh1`) is used to locate the corresponding entry in the person table. Compare exact age and bracketed age separately. A bracket is not treated as an exact age.


In [5]:
# Compare the derived reference-person age with the corresponding person record.
age_rows = []
required_d = ("sa0100", "sa0010", "im0100", "dhidh1", "dhageh1", "dhageh1b")
required_p = ("sa0100", "sa0010", "im0100", "ra0010", "ra0300", "ra0300_b")


def read_selected(wave, table, imp, wanted):
    path = parquet_path(wave, table, imp)
    mapping = clean_name_map(pq.ParquetFile(path).schema_arrow.names)
    available = [c for c in wanted if c in mapping]
    df = pd.read_parquet(path, columns=[mapping[c] for c in available])
    df.columns = available
    return df


for wave in WAVES:
    for imp in IMPUTATIONS:
        d = read_selected(wave, "d", imp, required_d)
        p = read_selected(wave, "p", imp, required_p)
        for column in required_d:
            if column not in d:
                d[column] = pd.NA
        for column in required_p:
            if column not in p:
                p[column] = pd.NA

        merge_key = ["sa0100", "sa0010", "im0100", "ra0010"]
        if d["dhidh1"].notna().any() and p["ra0010"].notna().any():
            d = d.rename(columns={"dhidh1": "ra0010"})
            merged = d.merge(p, on=merge_key, how="left", indicator=True,
                             validate="one_to_one", suffixes=("_d", "_p"))
        else:
            # Still report availability if no reference-person link can be constructed.
            d = d.rename(columns={"dhidh1": "ra0010"})
            merged = d.assign(ra0300=pd.NA, ra0300_b=pd.NA, _merge="left_only")

        for country, sub in country_groups(merged, "sa0100"):
            exact_d = sub["dhageh1"].notna()
            bracket_d = sub["dhageh1b"].notna()
            exact_p = sub["ra0300"].notna()
            bracket_p = sub["ra0300_b"].notna()
            matched = sub["_merge"].eq("both")
            compare_exact = exact_d & exact_p
            compare_bracket = bracket_d & bracket_p
            mismatch_exact = int((sub.loc[compare_exact, "dhageh1"].astype(float).values !=
                                  sub.loc[compare_exact, "ra0300"].astype(float).values).sum())
            mismatch_bracket = int((sub.loc[compare_bracket, "dhageh1b"].astype(float).values !=
                                    sub.loc[compare_bracket, "ra0300_b"].astype(float).values).sum())
            total = len(sub)
            age_rows.append({
                "wave": wave, "country": str(country), "imputation": imp,
                "households": total, "reference_person_linked": int(matched.sum()),
                "exact_derived_available": int(exact_d.sum()),
                "exact_person_available": int(exact_p.sum()),
                "bracket_derived_available": int(bracket_d.sum()),
                "bracket_person_available": int(bracket_p.sum()),
                "compared_exact": int(compare_exact.sum()),
                "exact_disagreements": mismatch_exact,
                "compared_bracket": int(compare_bracket.sum()),
                "bracket_disagreements": mismatch_bracket,
                "age_class": "exact" if exact_d.all() and total else (
                    "brackets_only" if not exact_d.any() and bracket_d.all() and total else "mixed_or_missing"),
            })
            if mismatch_exact or mismatch_bracket:
                issues.append({"wave": wave, "country": str(country), "table": "d+p",
                               "imputation": imp, "variable": "age",
                               "issue": "reference_person_age_mismatch",
                               "detail": f"exact={mismatch_exact}; bracket={mismatch_bracket}"})
        del d, p, merged
        gc.collect()

age_df = pd.DataFrame(age_rows)
print("Age checks:", len(age_df), "| Brackets-only country-imputation rows:",
      int(age_df.age_class.eq("brackets_only").sum()))


Age checks: 505 | Brackets-only country-imputation rows: 35


## Stata variable and value labels

Use the metadata exported from the original `.dta` files. Missing labels are left blank, not guessed from the variable name.


In [6]:
# Labels are saved alongside the Parquet files by the export notebook.
meta_dir = INPUT / "metadata"
variable_label_file = meta_dir / "variable_labels.csv"
value_label_file = meta_dir / "value_labels.csv"

def label_key(code):
    if pd.isna(code):
        return None
    text = str(code).strip()
    try:
        as_float = float(text)
        if np.isfinite(as_float) and as_float.is_integer():
            return str(int(as_float))
    except (TypeError, ValueError):
        pass
    return text


if variable_label_file.is_file():
    vlabels = pd.read_csv(variable_label_file, keep_default_na=False)
    vlabels["variable"] = vlabels["variable"].str.lower()
    vlabels["table"] = vlabels["table"].str.lower()
    vlabels = vlabels.loc[vlabels["variable_label"].astype(str).str.strip().ne("")]
    vlabels = (vlabels.sort_values("imputation")
               .drop_duplicates(["wave", "table", "variable"], keep="first")
               [["wave", "table", "variable", "variable_label"]])
else:
    vlabels = pd.DataFrame(columns=["wave", "table", "variable", "variable_label"])
    print("No variable_labels.csv found. Run the Stata label export cell.")

if value_label_file.is_file():
    clabels = pd.read_csv(value_label_file, keep_default_na=False)
    clabels["variable"] = clabels["variable"].str.lower()
    clabels["table"] = clabels["table"].str.lower()
    clabels["code_key"] = clabels["code"].map(label_key)
    clabels = (clabels.sort_values("imputation")
               .drop_duplicates(["wave", "table", "variable", "code_key"], keep="first")
               [["wave", "table", "variable", "code_key", "value_label"]])
else:
    clabels = pd.DataFrame(columns=["wave", "table", "variable", "code_key", "value_label"])
    print("No value_labels.csv found. Code values remain unlabelled.")

missing_df = missing_df.merge(vlabels, on=["wave", "table", "variable"], how="left", validate="many_to_one")
code_df = pd.DataFrame(code_rows, columns=[
    "wave", "country", "table", "variable", "raw_numeric_code", "count", "pct_of_nonmissing"])
code_df["code_key"] = code_df["raw_numeric_code"].map(label_key)
code_df = code_df.merge(vlabels, on=["wave", "table", "variable"], how="left", validate="many_to_one")
code_df = code_df.merge(clabels, on=["wave", "table", "variable", "code_key"], how="left", validate="many_to_one")
code_df = code_df.drop(columns="code_key")
print("Labelled variables:", len(vlabels), "| Labelled code values:", len(clabels))


Labelled variables: 1043 | Labelled code values: 0


## Reports and review flags

The private reports include counts and percentages. The repository reports contain only coarse missingness categories, age availability and selected structural warnings. Review the outputs before sharing; aggregation alone does not guarantee disclosure safety.


In [7]:
if missing_df.empty:
    missing_summary_df = pd.DataFrame()
else:
    missing_summary_df = (
        missing_df.groupby(["wave", "country", "table", "variable"], as_index=False)
        .agg(imputations_available=("imputation", "nunique"),
             rows_per_imputation_min=("rows", "min"),
             rows_per_imputation_max=("rows", "max"),
             mean_missing_pct=("missing_pct", "mean"),
             min_missing_pct=("missing_pct", "min"),
             max_missing_pct=("missing_pct", "max"),
             min_observed_n=("observed_n", "min"),
             max_observed_n=("observed_n", "max"),
             any_extended_stata_missing=("stata_extended_missing_n", lambda s: bool(s.gt(0).any())),
             any_empty_strings=("empty_string_n", lambda s: bool(s.gt(0).any())))
    )
    pct_cols = ["mean_missing_pct", "min_missing_pct", "max_missing_pct"]
    missing_summary_df[pct_cols] = missing_summary_df[pct_cols].round(3)
    missing_summary_df = missing_summary_df.merge(
        vlabels, on=["wave", "table", "variable"], how="left", validate="many_to_one")
    # The transfer-amount fields can be missing because nobody qualified for the question.
    # They remain in the report but are not automatically interpreted as data errors.
    watch = {"hh0100", "hh0201", "hh0401", "hh0501", "dhageh1",
             "dhageh1b", "dn3001", "ra0300", "ra0300_b"}
    all_absent = (missing_summary_df["min_missing_pct"].eq(100)
                  & missing_summary_df["variable"].isin(watch))
    for _, x in missing_summary_df.loc[all_absent].iterrows():
        issues.append({"wave": x.wave, "country": x.country, "table": x.table,
                       "imputation": "all", "variable": x.variable,
                       "issue": "fully_missing_all_imputations", "detail": "100% missing"})

if schema_df.empty:
    presence_df = pd.DataFrame()
else:
    real_schema = schema_df.loc[~schema_df["stata_missing_companion"]].copy()
    real_schema["variable"] = real_schema["variable"].str.lower()
    presence_df = (
        real_schema.groupby(["wave", "table", "variable"], as_index=False)
        .agg(imputations_available=("imputation", "nunique"),
             parquet_dtypes=("parquet_type", lambda s: "; ".join(sorted(set(s)))))
    )
    presence_df = presence_df.merge(
        vlabels, on=["wave", "table", "variable"], how="left", validate="many_to_one")

if not samples_df.empty:
    unknown = samples_df.loc[~samples_df.country.str.fullmatch("[A-Z]{2}")]
    for row in unknown[["wave", "country"]].drop_duplicates().itertuples():
        issues.append({"wave": row.wave, "country": row.country, "table": "all",
                       "imputation": "all", "variable": "sa0100",
                       "issue": "unusual_country_code", "detail": "Check the survey vintage / ECB documentation"})

# A collected-value flag on a completely missing variable merits review.
if not flag_df.empty and not missing_summary_df.empty:
    flagged = flag_df.loc[flag_df.meaning.eq("collected")].groupby(
        ["wave", "country", "table", "variable"], as_index=False).agg(
        imputations_with_collected_flag=("imputation", "nunique"))
    flagged["variable"] = flagged.variable.str.removeprefix("f")
    mismatch = missing_summary_df.merge(flagged, on=["wave", "country", "table", "variable"], how="inner")
    for row in mismatch.loc[mismatch.min_missing_pct.eq(100)].itertuples():
        issues.append({"wave": row.wave, "country": row.country, "table": row.table,
                       "imputation": "all", "variable": row.variable,
                       "issue": "collected_flag_but_no_values", "detail": "Check source flags / country metadata"})

issues_df = pd.DataFrame(issues, columns=["wave", "country", "table", "imputation", "variable", "issue", "detail"])

# Keep all counts outside Git. Small-cell disclosure review is required
# even for aggregated results (particularly the extended-missing detail).
private_reports = {
    "00_input_file_inventory.csv": files_df,
    "01_country_wave_counts.csv": samples_df,
    "02_variable_missingness_by_imp.csv": missing_df,
    "03_variable_missingness_summary.csv": missing_summary_df,
    "04_numeric_category_frequencies.csv": code_df,
    "05_stata_extended_missing_codes.csv": pd.DataFrame(extended_rows, columns=[
        "wave", "country", "table", "imputation", "variable", "stata_missing_code", "count"]),
    "06_h_d_identifier_matches.csv": joins_df,
    "07_issues_to_review.csv": issues_df,
    "08_variable_presence.csv": presence_df,
    "09_text_encoding_flags.csv": pd.DataFrame(text_rows, columns=[
        "wave", "country", "table", "imputation", "variable",
        "unicode_replacement_character_rows", "unicode_control_character_rows"]),
    "10_reference_age_check.csv": age_df,
    "11_target_flag_counts.csv": flag_df,
}

for filename, data in private_reports.items():
    data.to_csv(PRIVATE_OUTPUT / filename, index=False, encoding="utf-8-sig")

if WRITE_REPO_SUMMARIES:
    # No imputation-specific cell counts, small category counts or exact missing counts.
    if missing_df.empty:
        missing_public = pd.DataFrame()
    else:
        missing_public = missing_df[["wave", "country", "table", "imputation",
                                     "variable", "variable_label"]].copy()
        pct = missing_df["missing_pct"]
        missing_public["missing_status"] = np.select(
            [pct.eq(0), pct.eq(100)], ["none", "all"], default="partial")

    if missing_summary_df.empty:
        summary_public = pd.DataFrame()
    else:
        summary_public = missing_summary_df[["wave", "country", "table", "variable",
                                             "variable_label", "imputations_available"]].copy()
        summary_public["missing_status"] = np.select(
            [missing_summary_df.max_missing_pct.eq(0),
             missing_summary_df.min_missing_pct.eq(100)],
            ["none", "all"], default="partial")

    age_public = age_df[["wave", "country", "imputation", "age_class"]].copy()
    age_public["reference_person_match_status"] = np.where(
        age_df.reference_person_linked.eq(age_df.households), "complete", "incomplete")
    age_public["exact_age_consistency"] = np.where(age_df.exact_disagreements.eq(0), "no_conflict", "review")
    age_public["bracket_consistency"] = np.where(age_df.bracket_disagreements.eq(0), "no_conflict", "review")

    # Make flag semantics visible but not the count for each code/category.
    flag_public = flag_df.loc[flag_df["count"].ge(MIN_FREQUENCY_TO_PUBLISH),
                              ["wave", "country", "table", "imputation", "variable", "flag", "meaning"]].copy()
    flag_public = flag_public.drop_duplicates()

    safe_reports = {
        "00_input_file_inventory.csv": files_df,
        "01_country_wave_counts.csv": samples_df[["wave", "country", "table", "imputation", "rows"]].loc[
            samples_df.rows.ge(MIN_FREQUENCY_TO_PUBLISH)],
        "02_variable_missingness_by_imp.csv": missing_public,
        "03_variable_missingness_summary.csv": summary_public,
        "04_numeric_category_frequencies.csv": private_reports["04_numeric_category_frequencies.csv"],
        "05_stata_extended_missing_codes.csv": pd.DataFrame(columns=[
            "wave", "country", "table", "imputation", "variable", "stata_missing_code", "count"]),
        "06_h_d_identifier_matches.csv": joins_df[["wave", "country", "imputation"]].copy(),
        "07_issues_to_review.csv": issues_df[["wave", "country", "table", "imputation", "variable", "issue"]].copy(),
        "08_variable_presence.csv": presence_df,
        "09_text_encoding_flags.csv": pd.DataFrame(columns=[
            "wave", "country", "table", "imputation", "variable"]),
        "10_reference_age_check.csv": age_public,
        "11_target_flag_counts.csv": flag_public,
    }
    for filename, data in safe_reports.items():
        data.to_csv(PUBLIC_OUTPUT / filename, index=False, encoding="utf-8-sig")
    (PUBLIC_OUTPUT / "README.txt").write_text(
        "HFCS diagnostic summaries (no household microdata).\n"
        "Full counts and missingness rates are kept locally in HFCS_selected_raw/diagnostics_private.\n"
        "The repo versions are coarse or omit low-frequency information.\n"
        "03 reports none/partial/all missingness, not the missing rate.\n"
        "Where available, 02-04 and 08 include labels from the Stata files.\n"
        "05 and 09 deliberately contain no detailed counts.\n"
        "Flags: 2017 code 2050 = anonymisation and 2051 = not collected.\n"
        "For 2021/2023, code 2050 only means set missing; reason is unspecified.\n"
        "Confirm any release against ECB disclosure requirements before publishing.\n",
        encoding="utf-8")

print("Detailed reports:", PRIVATE_OUTPUT)
print("Repo summaries:", PUBLIC_OUTPUT if WRITE_REPO_SUMMARIES else "not written")
print("Issues flagged:", len(issues_df))
if not issues_df.empty:
    display(issues_df.issue.value_counts().rename_axis("issue").reset_index(name="rows"))


Detailed reports: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester II\RA\HFCS Data Download\HFCS_selected_raw\diagnostics_private
Repo summaries: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester II\RA\Research-Assistant-Git\HFCS\diagnostics
Issues flagged: 143


,issue,rows
0,reference_person_age_mismatch,80
1,fully_missing_all_imputations,60
2,collected_flag_but_no_values,2
3,unusual_country_code,1


In [8]:
# Export variable dictionary for manual review

labels = pd.read_csv(
    INPUT / "metadata" / "variable_labels.csv"
)

labels["variable"] = labels["variable"].str.lower()
labels["variable_label"] = labels["variable_label"].fillna("").str.strip()

# One record per variable, wave and table
dictionary = (
    labels.groupby(["wave", "table", "variable"], as_index=False)
    .agg(
        variable_label=("variable_label", lambda x: " | ".join(sorted(set(x)))),
        imputations_available=("imputation", "nunique"),
        labels_within_wave=("variable_label", "nunique")
    )
)

# Identify variables whose descriptions differ across waves
dictionary["labels_across_waves"] = (
    dictionary.groupby(["table", "variable"])["variable_label"]
    .transform("nunique")
)

dictionary["review_label"] = (
    dictionary["variable_label"].eq("")
    | dictionary["labels_within_wave"].gt(1)
    | dictionary["labels_across_waves"].gt(1)
)

# Add data types from the existing diagnostics
dictionary = dictionary.merge(
    presence_df[
        ["wave", "table", "variable", "parquet_dtypes"]
    ],
    on=["wave", "table", "variable"],
    how="left"
)

dictionary = dictionary[
    [
        "wave",
        "table",
        "variable",
        "variable_label",
        "parquet_dtypes",
        "imputations_available",
        "review_label"
    ]
].sort_values(["table", "variable", "wave"])

output = PUBLIC_OUTPUT / "12_variable_dictionary.csv"
dictionary.to_csv(output, index=False, encoding="utf-8-sig")

print(f"Saved: {output}")
print(f"Variables: {len(dictionary)}")
print(f"Entries requiring review: {dictionary.review_label.sum()}")

display(dictionary[dictionary.review_label].head(30))

Saved: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester II\RA\Research-Assistant-Git\HFCS\diagnostics\12_variable_dictionary.csv
Variables: 1108
Entries requiring review: 311


,wave,table,variable,variable_label,parquet_dtypes,imputations_available,review_label
49,2010,d,datop10,DATOP10 Country top 10% gross wealth,int8,5,True
258,2014,d,datop10,DATOP10 Country top 10% gross wealth,int8,5,True
477,2017,d,datop10,DATOP10 Country top 10% gross wealth,int8,5,True
704,2021,d,datop10,DATOP10 Country top 10% gross wealth,int8,5,True
930,2023,d,datop10,"DATOP10 Percentiles, gross wealth, among house...",int8,5,True
50,2010,d,datop10ea,DATOP10EA EA top 10% gross wealth,int64,5,True
259,2014,d,datop10ea,DATOP10EA EA top 10% gross wealth,int8,5,True
478,2017,d,datop10ea,DATOP10EA EA top 10% gross wealth,int64,5,True
705,2021,d,datop10ea,DATOP10EA EA top 10% gross wealth,int64,5,True
931,2023,d,datop10ea,"DATOP10EA EA percentiles, gross wealth, among ...",int64,5,True


## Notes for the estimation stage

- Use `dhageh1` where available. Where only `dhageh1b` is available, analyse age in groups, not as exact single-year ages.
- `hh0100` covers substantial gifts **and** inheritances. A missing response is not a zero transfer.
- The HFCS flag codes depend on the wave. In 2017, `2050` means anonymisation and `2051` means not collected. In 2021 and 2023, `2050` indicates a value set to missing without identifying the reason.
- Italy's `hw0010a` is retained for historical comparisons. The correct weights depend on the estimation exercise.
- Treat `E1` in 2010 as a survey-vintage code to verify, not as a separate country.
- The notebook does not pool imputations or alter the raw data.
